In [1]:
import pandas as pd

roll_number = "1024170150"

fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

last_two_digits = [int(d) for d in roll_number[-2:]]

categories = ["billing", "account", "general"]

personalized_1 = {
    "question": "how can i contact support",
    "answer": "You can contact support through the help desk.",
    "keywords": "support help contact",
    "category": categories[last_two_digits[0] % 3]
}

personalized_2 = {
    "question": "how can i check my payment status",
    "answer": "You can check your payment status in the billing section.",
    "keywords": "payment status billing",
    "category": categories[last_two_digits[1] % 3]
}

faq_data = fixed_entries + [personalized_1, personalized_2]

df = pd.DataFrame(faq_data)

print(df)

                            question  \
0             what is the annual fee   
1              how to reset password   
2        what are your working hours   
3              how can i pay the fee   
4          how can i contact support   
5  how can i check my payment status   

                                              answer                keywords  \
0                          The annual fee is Rs 500.   fee cost price charge   
1                   Go to Settings > Reset Password.    password reset login   
2                          We are open 9 AM to 5 PM.  hours timing open time   
3         You can pay via UPI, card, or net banking.     pay payment upi fee   
4     You can contact support through the help desk.    support help contact   
5  You can check your payment status in the billi...  payment status billing   

  category  
0  billing  
1  account  
2  general  
3  billing  
4  general  
5  billing  


In [2]:
def score_query(query, df):
    query_words = set(query.lower().split())

    results = []

    for index, row in df.iterrows():
        keywords = set(row["keywords"].lower().split())

        matched_words = query_words.intersection(keywords)
        score = len(matched_words)

        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })

    results = sorted(results, key=lambda x: x["score"], reverse=True)

    return pd.DataFrame(results)


query = input("Enter your question: ")

result = score_query(query, df)

print("\nMatching FAQs:")
print(result)

Enter your question: what is the annual fee

Matching FAQs:
   index                question                                      answer  \
0      0  what is the annual fee                   The annual fee is Rs 500.   
1      3   how can i pay the fee  You can pay via UPI, card, or net banking.   

  category  score  
0  billing      1  
1  billing      1  


In [3]:
def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()]

category_name = personalized_1["category"]

print("Category:", category_name)
print("\nFAQs in this category:")
print(same_category(category_name, df))

Category: general

FAQs in this category:
                      question  \
2  what are your working hours   
4    how can i contact support   

                                           answer                keywords  \
2                       We are open 9 AM to 5 PM.  hours timing open time   
4  You can contact support through the help desk.    support help contact   

  category  
2  general  
4  general  


In [4]:
print(df[["question", "keywords"]])

entry_index = int(input("\nEnter the index of the FAQ entry to update (0-5): "))

new_keyword = input("Enter a new keyword: ")

df.loc[entry_index, "keywords"] = (
    df.loc[entry_index, "keywords"] + " " + new_keyword
)

print("\nUpdated DataFrame:")
print(df)

filename = "1024170150_faq_data.csv"
df.to_csv(filename, index=False)

print("\nFile saved as:", filename)

                            question                keywords
0             what is the annual fee   fee cost price charge
1              how to reset password    password reset login
2        what are your working hours  hours timing open time
3              how can i pay the fee     pay payment upi fee
4          how can i contact support    support help contact
5  how can i check my payment status  payment status billing

Enter the index of the FAQ entry to update (0-5): 4
Enter a new keyword: support

Updated DataFrame:
                            question  \
0             what is the annual fee   
1              how to reset password   
2        what are your working hours   
3              how can i pay the fee   
4          how can i contact support   
5  how can i check my payment status   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                     

In [5]:
category_counts = df.groupby("category").size()

print(category_counts)

category
account    1
billing    3
general    2
dtype: int64


In [6]:
def score_query_with_ties(query, df):
    query_words = set(query.lower().split())

    results = []

    for index, row in df.iterrows():
        keywords = set(row["keywords"].lower().split())

        matched_words = query_words.intersection(keywords)
        score = len(matched_words)

        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })

    if len(results) == 0:
        print("No matching FAQ found.")
        return

    # Find highest score
    highest_score = max(item["score"] for item in results)

    # Keep all entries with highest score
    best_matches = [
        item for item in results
        if item["score"] == highest_score
    ]

    print("\nHighest confidence score:", highest_score)

    if len(best_matches) > 1:
        print("Tie detected! Multiple FAQs have the same highest score.")
    else:
        print("No tie. One FAQ has the highest score.")

    result_df = pd.DataFrame(best_matches)
    print("\nBest matching FAQ(s):")
    print(result_df)

    return result_df

In [7]:
print("TIE EXAMPLE")
score_query_with_ties("fee", df)

TIE EXAMPLE

Highest confidence score: 1
Tie detected! Multiple FAQs have the same highest score.

Best matching FAQ(s):
   index                question                                      answer  \
0      0  what is the annual fee                   The annual fee is Rs 500.   
1      3   how can i pay the fee  You can pay via UPI, card, or net banking.   

  category  score  
0  billing      1  
1  billing      1  


,index,question,answer,category,score
0,0,what is the annual fee,The annual fee is Rs 500.,billing,1
1,3,how can i pay the fee,"You can pay via UPI, card, or net banking.",billing,1


In [8]:
print("\nNON-TIE EXAMPLE")
score_query_with_ties("password reset", df)


NON-TIE EXAMPLE

Highest confidence score: 2
No tie. One FAQ has the highest score.

Best matching FAQ(s):
   index               question                            answer category  \
0      1  how to reset password  Go to Settings > Reset Password.  account   

   score  
0      2  


,index,question,answer,category,score
0,1,how to reset password,Go to Settings > Reset Password.,account,2
